# QMaxCal quickstart

A short tour of this repository using the amplitude-damped qubit from Section 3
of the paper (task |+⟩ → |Y⟩, γT = 2). It shows how to

1. build a system,
2. load a saved controller from `results/`,
3. evaluate it with the exact Lindblad solver and with SSE trajectories,
4. summarise a folder of saved runs,
5. launch a small training run.

Everything here uses saved results except step 5, which trains for a few hundred
steps (about a minute on a GPU, a few minutes on a CPU). Run the notebook from
the repository root.

In [ ]:
import os, sys, glob, pickle, re, tempfile
sys.path.insert(0, os.path.abspath("."))
assert os.path.exists("train_qmaxcal.py"), "start Jupyter from the repository root"

import numpy as np
import jax, jax.numpy as jnp
from flax import nnx
import matplotlib.pyplot as plt

from core.types import ExponentialSplit, asqarray
from core.solver import dssesolve
import train_qmaxcal as tq
from systems import amplitude_damping

print("JAX devices:", jax.devices())

## 1. Systems

Each file in `systems/` exposes `create_system(**kwargs)`, which returns a plain
dict: drift Hamiltonian `H_sys`, control operators `H_ops`, jump operators
`jump_ops`, initial and target states, the horizon `T_final`, and the
projectors used for plotting (`populations`). The trainer only relies on this
dict, so adding a new system means writing one such file.

In [ ]:
GAMMA = 2.0
sysd = amplitude_damping.create_system(gamma=GAMMA, T_final=1.0)
for k in ["system_name", "T_final", "n_controls", "control_labels"]:
    print(f"{k:15s}", sysd[k])
print("jump operator L =\n", np.asarray(sysd["jump_ops"][0]))
print("populations:", list(sysd["populations"]))

## 2. Load a saved controller

The saved controllers behind the paper are in `paper/results/`. Training writes one folder per run, named after the system and the loss
weights, e.g. `ampdamp_g2.0_T1.0_kl5.0_dv0.0_fl0.01_seed1`
(`kl` = Wiener-KL weight λ_KLW, `dv` = drift-variance weight, `fl` = fluence weight).
Each folder holds

* `controller.pkl`: Fourier coefficients **at the last training step**, plus
  `best_exact_fidelity` / `best_step` of the best exact-Lindblad snapshot,
* `controller_best_by_loss.pkl` (newer runs only): coefficients at the step with
  the lowest training loss, plus their exact fidelity `f_exact_at_best_loss`.

The controller is a truncated Fourier series per control:
u(t) = dc + Σ_{n=1}^{N} [a_n cos(2πnt/T) + b_n sin(2πnt/T)].

In [ ]:
def load_controller(path):
    # Rebuild a FourierController from a saved checkpoint.
    with open(path, "rb") as f:
        ck = pickle.load(f)
    n_ctrl, n_modes = np.shape(ck["best_cos_coeffs"])
    ctrl = tq.FourierController(n_controls=n_ctrl, n_modes=n_modes,
                                T_final=float(ck["T_final"]), rngs=nnx.Rngs(0))
    ctrl.dc         = nnx.Param(jnp.asarray(ck["best_dc"]))
    ctrl.cos_coeffs = nnx.Param(jnp.asarray(ck["best_cos_coeffs"]))
    ctrl.sin_coeffs = nnx.Param(jnp.asarray(ck["best_sin_coeffs"]))
    return ctrl, ck

BASE = "paper/results/ampdamp/qmaxcal/multiseed_g2.0"
RUNS = {
    "baseline (λ_KLW=0)": f"{BASE}/ampdamp_g2.0_T1.0_kl0.0_dv0.0_fl0.01_seed1/controller.pkl",
    "Wiener KL (λ_KLW=5)": f"{BASE}/ampdamp_g2.0_T1.0_kl5.0_dv0.0_fl0.01_seed1/controller.pkl",
}
ctrls = {name: load_controller(p) for name, p in RUNS.items()}

t = jnp.linspace(0, sysd["T_final"], 400)
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for ax, (name, (ctrl, ck)) in zip(axes, ctrls.items()):
    u = jax.vmap(ctrl)(t)
    for i, lab in enumerate(sysd["control_labels"]):
        ax.plot(t, u[:, i], label=lab)
    ax.set_title(name); ax.set_xlabel("t / T")
axes[0].set_ylabel("control amplitude"); axes[0].legend()
plt.tight_layout(); plt.show()

## 3a. Exact Lindblad fidelity

`tq.lindblad_trajectory` integrates the master equation with RK4 (300 steps)
and returns the population curves and the final density matrix. This is the
fidelity reported in the paper. When `best_step` is the last step the
recomputed value matches `best_exact_fidelity`; otherwise it is the
fidelity of the final-step coefficients.

In [ ]:
def exact_fidelity(ctrl, sysd):
    times, pops, rho = tq.lindblad_trajectory(ctrl, sysd)
    tgt = sysd["target_state"]
    F = float(jnp.real(jnp.trace((tgt @ tgt.conj().T) @ rho)))
    return F, times, pops

fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for ax, (name, (ctrl, ck)) in zip(axes, ctrls.items()):
    F, times, pops = exact_fidelity(ctrl, sysd)
    print(f"{name:22s} F_exact(saved coeffs) = {F:.4f}   "
          f"best snapshot = {ck['best_exact_fidelity']:.4f} at step {ck['best_step']}")
    for lab, p in pops.items():
        ax.plot(times, p, label=lab)
    ax.set_title(f"{name}\nF = {F:.4f}"); ax.set_xlabel("t / T")
axes[0].set_ylabel("population"); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 3b. SSE trajectories and the regularizers

Training does not use the Lindblad solver. It samples diffusive quantum
trajectories with `core.solver.dssesolve` and differentiates through them.
`tq.compute_loss` returns the trajectory-averaged fidelity and every
regularizer value, whatever weights you pass:

* `kl_cost`: Wiener KL, ½ E[∫ Σ_k α_k² dt] with record drift α_k = 2 Re⟨L_k⟩,
* `drift_var_cost`: R_DV, the time variance of α,
* `var_l_cost`: Var(L) = E[∫ (⟨L†L⟩ − |⟨L⟩|²) dt], LO-phase invariant,
* `fluence`: ∫ |u|² dt.

The Wiener-KL controller trades a little fluence for a much smaller KL and a
narrower spread of trajectories.

In [ ]:
N_TRAJ, N_STEPS = 256, 256
sys_q = tq._wrap_system(sysd)
tsave = jnp.linspace(0, sysd["T_final"], N_STEPS + 1)
keys  = jax.random.split(jax.random.PRNGKey(123), N_TRAJ)

print(f"{'':22s} {'F_SSE':>7s} {'KL_W':>7s} {'R_DV':>7s} {'Var(L)':>7s} {'fluence':>8s}")
for name, (ctrl, _) in ctrls.items():
    _, m = tq.compute_loss(ctrl, sys_q, keys, tsave, N_STEPS)
    print(f"{name:22s} {float(m['fidelity']):7.4f} {float(m['kl_cost']):7.4f} "
          f"{float(m['drift_var_cost']):7.4f} {float(m['var_l_cost']):7.4f} {float(m['fluence']):8.3f}")

In [ ]:
# Plot individual trajectories of the target population |<Y|psi_t>|^2.
tgt = np.asarray(sysd["target_state"])[:, 0]
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for ax, (name, (ctrl, _)) in zip(axes, ctrls.items()):
    res = dssesolve(H_sys=sys_q["H_sys"], jump_ops=sys_q["jump_ops"], psi0=sys_q["psi0"],
                    tsave=tsave, keys=keys, H_con=tq.build_H_con(ctrl, sys_q["H_ops"]),
                    method=ExponentialSplit(dt=sysd["T_final"] / N_STEPS),
                    nsteps=N_STEPS, nsteps_per_save=1)
    psi = np.asarray(res.states.data)[..., 0]           # (traj, time, dim)
    pY  = np.abs(psi @ tgt.conj()) ** 2
    ax.plot(tsave, pY[:40].T, color="C0", alpha=0.25, lw=0.8)
    ax.plot(tsave, pY.mean(0), color="k", lw=2, label="mean")
    ax.set_title(name); ax.set_xlabel("t / T")
axes[0].set_ylabel(r"$|\langle Y|\psi_t\rangle|^2$"); axes[0].legend()
plt.tight_layout(); plt.show()

## 4. Summarise a folder of runs

The settings of a run can be read from its folder name, so a results table is a
short loop. Here: all γT = 2 runs used for Section 3 (mean ± std over seeds of
the best exact-Lindblad snapshot).

In [ ]:
rows = {}
for p in sorted(glob.glob(f"{BASE}/*/controller.pkl")):
    run = p.split("/")[-2]
    kl, dv, seed = re.search(r"_kl([\d.]+)_dv([\d.]+)_.*seed(\d+)", run).groups()
    with open(p, "rb") as f:
        rows.setdefault((float(kl), float(dv)), []).append(pickle.load(f)["best_exact_fidelity"])

print(f"{'λ_KLW':>6s} {'λ_DV':>6s}  n   F_exact")
for (kl, dv), fs in sorted(rows.items()):
    print(f"{kl:6.1f} {dv:6.1f}  {len(fs)}   {np.mean(fs):.4f} ± {np.std(fs, ddof=1) if len(fs) > 1 else 0:.4f}")

## 5. Train a small run

`tq.train` is what `python train_qmaxcal.py ...` calls. This demo uses far fewer
steps and trajectories than the paper (5000 steps, 256 trajectories), so expect
a lower fidelity. The equivalent command line is

```bash
python train_qmaxcal.py --system systems.amplitude_damping --gamma 2.0 \
    --kl_weight 5.0 --fluence_weight 0.01 --n_modes 20 --init_scale 0.1 \
    --n_trajs 64 --num_steps 300 --snap_every 100 --output_dir <dir>
```

Every system module adds its own flags (`--gamma`, `--lo_phase`, ...) through
`add_system_args`. Run `python train_qmaxcal.py --system systems.<name> --help` to see them.

In [ ]:
out_dir = tempfile.mkdtemp(prefix="qmaxcal_demo_")
tq.train(amplitude_damping, dict(gamma=2.0, T_final=1.0), out_dir,
         kl_weight=5.0, fluence_weight=0.01,
         n_time_steps=256, n_trajs=64, num_steps=300, snap_every=100,
         lr=1e-3, n_modes=20, init_scale=0.1, seed=0)

run_dir = glob.glob(f"{out_dir}/*")[0]
print(run_dir); print(sorted(os.listdir(run_dir)))
ctrl_demo, ck_demo = load_controller(f"{run_dir}/controller_best_by_loss.pkl")
print("F_exact at best training loss:", round(ck_demo["f_exact_at_best_loss"], 4))

## 6. Next steps

* The main `README.md` lists every training option and explains how to add a new system.
* `paper/README.md` explains how to regenerate each figure and table of the paper.